# Lab 6: Complex Data Structures in Spark 
## Nested Data Processing with User-Defined Functions

Today's lab focuses on working with complex data structures in Spark, specifically nested data. Often, data is not flat and may contain nested fields such as arrays, structs, or maps. Spark provides powerful tools to handle such data. After working on the data loading together, it is up to you to follow the instructions and complete the exercise. 


### Installing Spark

In [37]:
#Checking the installed Java version
!java -version

openjdk version "17.0.16" 2025-07-15
OpenJDK Runtime Environment (build 17.0.16+8-Ubuntu-0ubuntu124.04.1)
OpenJDK 64-Bit Server VM (build 17.0.16+8-Ubuntu-0ubuntu124.04.1, mixed mode, sharing)


In [38]:
!pip install pyspark 

In [39]:
# Install Java 17
!sudo apt-get update
!sudo apt-get install -y openjdk-17-jdk-headless


Hit:1 https://packages.cloud.google.com/apt cloud-sdk InRelease
Hit:2 https://cli.github.com/packages stable InRelease                         
Hit:3 https://download.docker.com/linux/ubuntu noble InRelease                 
Hit:4 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble InRelease          
Hit:5 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates InRelease  
Hit:6 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:7 https://archive.ubuntu.com/ubuntu noble InRelease                        
Get:8 https://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]       
Get:9 https://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]     
Hit:10 http://deb.wakemeops.com/wakemeops stable InRelease                     
Get:11 https://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Hit:12 https://cloud.archive.ubuntu.com/ubuntu noble InRelease                 
Get:13 https://archive.ubuntu.com/ubuntu noble-updates/main 

In [40]:
!java -version

openjdk version "17.0.16" 2025-07-15
OpenJDK Runtime Environment (build 17.0.16+8-Ubuntu-0ubuntu124.04.1)
OpenJDK 64-Bit Server VM (build 17.0.16+8-Ubuntu-0ubuntu124.04.1, mixed mode, sharing)


In [41]:

# Set JAVA_HOME to Java 17
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"


In [47]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
        .master("local[*]")\
        .appName("Planetary") \
        .getOrCreate()
print("Spark ready:", spark.version)


Spark ready: 4.0.1


25/10/20 17:40:35 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [45]:
# Importing functions and types
from pyspark.sql import types as tp


#### Reading the planets dataset

In [ ]:
# Define the file path
# file_path # Define the file path



+------------------------------------+------------------------+---------------------------------------+-------------------+------------------+---------------+-----------------+----------------------+
|uuid                                |system                  |stardate_discovery                     |flybys             |habitability_index|elements       |planet_taxonomy  |planetary_temperatures|
+------------------------------------+------------------------+---------------------------------------+-------------------+------------------+---------------+-----------------+----------------------+
|1856a72e-2fde-4f05-b5b5-641f860b8bea|Proxima Centauri Cluster|4484.736272674565-m/0.3601666816523666 |"[[""WXY-234""]    |[""STU-234""]     |[""VWX-98765""]|[""MNO-1234""]   |[""CDE-543""]]"       |
|33dd1b1a-cf51-434c-9d4e-91e1dcf247a5|HD 209458 Formation     |4755.956377260051-s/0.26318120298987957|"[[""WXY-12345""]]"|0.4966529856392779|"[""Ytterbium""|""Yttrium""      |""Beryllium""         |


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, ArrayType
)

# --- 0) Read CSV with robust quote handling (prevents column shifting) ---
csv_schema = "uuid string, system string, stardate_discovery string, flybys string, habitability_index string, elements string, planet_taxonomy string, planetary_temperatures string"

df = (spark.read
      .option("header", "true")
      .option("multiLine", "true")                     # if any rows span lines
      .option("quote", "\"")
      .option("escape", "\"")
      .option("unescapedQuoteHandling", "BACK_TO_DELIMITER")  # key for messy quotes
      .schema(csv_schema)
      .csv(file_path))

# --- 1) Define target nested schemas ---
# flybys_schema = 
# elements_schema = 
# planet_taxonomy_schema = 
# planetary_temps_schema = 


# --- 2) Minimal normaliser: trim + collapse doubled quotes introduced by CSV ---
def normalize_json(col):
    c = F.trim(col)
    # Turn [""ABC""] -> ["ABC"], and remove stray trailing/leading quotes/brackets artifacts
    c = F.regexp_replace(c, r'""', '"')
    return c

# --- 3) Safely coerce the numeric column (handles values like ["0.87"] or bad tokens) ---
NUM = r'([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)'
# obtain habitability_num by trying to directly cast the column to double, which will yield null for non-numeric values
# habitability_num = 
# fallback to extracting the first numeric token from the string using regex, which handles cases like ["0.87"] or "0.87 (estimated)"
# habitability_fallback = 
habitability_final = F.coalesce(habitability_num, habitability_fallback)

# --- 4) Parse JSON-ish columns into proper types ---
# planetary_df = 



root
 |-- uuid: string (nullable = true)
 |-- system: string (nullable = true)
 |-- stardate_discovery: string (nullable = true)
 |-- flybys: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: string (containsNull = true)
 |-- habitability_index: double (nullable = true)
 |-- elements: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- planet_taxonomy: struct (nullable = true)
 |    |-- taxonomy_level1: string (nullable = true)
 |    |-- taxonomy_level2: string (nullable = true)
 |    |-- confidence_level: double (nullable = true)
 |-- planetary_temperatures: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- measurement8: double (nullable = true)
 |    |    |-- measurement4: double (nullable = true)
 |    |    |-- measurement5: double (nullable = true)
 |    |    |-- measurement3: double (nullable = true)
 |    |    |-- measurement7: double (nullable = true)
 |    |    |-- measure

In [60]:
# Print the schema of the DataFrame
planetary_df.show(5)

+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+----------------------+
|                uuid|              system|  stardate_discovery|              flybys| habitability_index|            elements|     planet_taxonomy|planetary_temperatures|
+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+----------------------+
|1856a72e-2fde-4f0...|Proxima Centauri ...|4484.736272674565...|[[WXY-234], [STU-...|  0.648708376148531|[Platinum, Hafniu...|{savannah, dry, 0...|  [{30.301, 37.108,...|
|33dd1b1a-cf51-434...| HD 209458 Formation|4755.956377260051...|       [[WXY-12345]]| 0.4966529856392779|[Ytterbium, Yttri...|{arctic, frozen, ...|  [{-3.143, 1.928, ...|
|50bcbce3-1776-498...| HD 209458 Formation|1794.713080699246...|[[ZAB-6789], [TUV...|0.47792946887426335|  [Terbium, Yttrium]|{desert, dry, 0.704

In [53]:
planetary_df_fixed.printSchema()

root
 |-- uuid: string (nullable = true)
 |-- system: string (nullable = true)
 |-- stardate_discovery: string (nullable = true)
 |-- flybys: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: string (containsNull = true)
 |-- habitability_index: double (nullable = true)
 |-- elements: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- planet_taxonomy: struct (nullable = true)
 |    |-- taxonomy_level1: string (nullable = true)
 |    |-- taxonomy_level2: string (nullable = true)
 |    |-- confidence_level: double (nullable = true)
 |-- planetary_temperatures: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- measurement8: double (nullable = true)
 |    |    |-- measurement4: double (nullable = true)
 |    |    |-- measurement5: double (nullable = true)
 |    |    |-- measurement3: double (nullable = true)
 |    |    |-- measurement7: double (nullable = true)
 |    |    |-- measure

Note that the column `planet_taxonomy` is a complex data structure: an `struct` with the information of each taxonomy element.

In PySpark, a complex data structure refers to a data type that can store multiple values, often with varying data types or nested structures. Complex data structures in PySpark include:  
1. `Arrays`: An array is an ordered collection of elements, where each element can be of any data type, including other complex data types. In PySpark, arrays are represented using the ArrayType class.  

2. `Structs`: A struct is a collection of named fields, where each field can have a different data type, including other complex data types. Structs are similar to rows in a table or objects in a programming language. In PySpark, structs are represented using the StructType class.  

3. `Maps`: A map is a collection of key-value pairs, where keys are unique and both keys and values can be of any data type, including other complex data types. Maps are useful for representing associative arrays, dictionaries, or hash maps. In PySpark, maps are represented using the MapType class.  

These complex data structures allow you to work with more sophisticated data in your PySpark applications, such as nested JSON data or hierarchical data. You can manipulate complex data structures using built-in PySpark functions, as well as user-defined functions (UDFs) when necessary.

#### Reading messy CSV into nested columns (what the code is doing, step-by-step)

**Why this is needed.** Our CSV stores *nested* data (arrays and structs) as text that looks like JSON, but the quotes are messy (e.g., `[""ABC""]`). If we read it naively, columns can shift or the JSON fails to parse. So we: **read safely → clean strings → parse into real nested types → fix numbers**.

**1) Safe CSV read (keep columns aligned).**
We pass a simple **top-level schema** (strings for the JSON-ish columns) and enable a few CSV options to tolerate bad quotes and multiline rows. This prevents “column drift” during parsing and skips slow type inference. Spark’s CSV reader supports options like `quote`, `escape`, `multiLine`, and `unescapedQuoteHandling` (e.g., `BACK_TO_DELIMITER` or `STOP_AT_CLOSING_QUOTE`) to handle unescaped quotes robustly. 

**2) Normalise the JSON-ish strings.**
Before we can parse, we make the text into **valid JSON**. We use `trim` and `regexp_replace` to collapse doubled quotes (`""` → `"`). These are standard column expressions in PySpark and run lazily as part of the DataFrame plan. 

**3) Parse strings into real nested columns.**
With clean JSON text, we call `from_json(col, schema)` and supply the **exact nested schema** we want (e.g., `array<array<string>>` for `flybys`, a `StructType` with named fields for `planet_taxonomy`, and an array of structs for `planetary_temperatures`). `from_json` materialises true Spark types (`array`, `struct`) and returns `NULL` for rows that still can’t be parsed. 

**4) Safely coerce the numeric column.**
Casting directly with ANSI semantics will fail on malformed values (e.g., `["0.87"]`). We therefore use `try_cast` to return `NULL` instead of throwing, and as a fallback we `regexp_extract` the first numeric token from the string (handles wrapped numbers like `["0.49665"]`) and cast that. This is the recommended pattern under ANSI mode. 

**5) Putting it together with `withColumn`.**
Each transformation (`from_json`, `try_cast`, regex cleanup) is applied via `withColumn`, which either creates or replaces a column with a new expression. Chaining these steps produces a DataFrame whose schema now has real **arrays** and **structs** that you can `select`, `explode`, aggregate, or save to Parquet/Delta. 

**Key takeaways.**

* CSV is **flat**; treat nested content as strings first, then parse to nested Spark types with `from_json` and an explicit schema.
* Use CSV options to avoid column misalignment when quotes are messy; `unescapedQuoteHandling` is especially helpful.
* Under ANSI, prefer `try_cast` (and regex fallback) for resilient numeric parsing.

Once parsed, these complex columns behave like any other Spark columns: you can reach into structs (`col("planet_taxonomy.confidence_level")`), index arrays, or `explode` them for row-wise operations.


#### Step 1: Extract the first probe that inspected each planet

In [ ]:
# TODO

+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+----------------------+------------+
|                uuid|              system|  stardate_discovery|              flybys| habitability_index|            elements|     planet_taxonomy|planetary_temperatures|first_flybys|
+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+----------------------+------------+
|1856a72e-2fde-4f0...|Proxima Centauri ...|4484.736272674565...|[[WXY-234], [STU-...|  0.648708376148531|[Platinum, Hafniu...|{savannah, dry, 0...|  [{30.301, 37.108,...|     WXY-234|
|33dd1b1a-cf51-434...| HD 209458 Formation|4755.956377260051...|       [[WXY-12345]]| 0.4966529856392779|[Ytterbium, Yttri...|{arctic, frozen, ...|  [{-3.143, 1.928, ...|   WXY-12345|
|50bcbce3-1776-498...| HD 209458 Formation|1794.713080699246...|[[ZAB-6789], [TU

#### Step 2: Extract only the daytime temperatures from each row

In [ ]:
# Create an array of the first 5 measurements
# assume that the daytime temperatures are stored in the first 5 fields of the "planetary_temperatures" struct


+--------------------+-------------------+
|                uuid|       daytime_mean|
+--------------------+-------------------+
|1856a72e-2fde-4f0...|            34.3137|
|33dd1b1a-cf51-434...|-0.3882999999999999|
|50bcbce3-1776-498...|            40.1674|
|729cb3af-41db-406...|            -1.3382|
|1dc8fffb-862e-4e9...|             30.738|
+--------------------+-------------------+



#### Step 3: Count how many rare elements there are on each planet

In [ ]:
# todo

+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+----------------------+-----------------+
|                uuid|              system|  stardate_discovery|              flybys| habitability_index|            elements|     planet_taxonomy|planetary_temperatures|num_rare_elements|
+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+----------------------+-----------------+
|1856a72e-2fde-4f0...|Proxima Centauri ...|4484.736272674565...|[[WXY-234], [STU-...|  0.648708376148531|[Platinum, Hafniu...|{savannah, dry, 0...|  [{30.301, 37.108,...|                4|
|33dd1b1a-cf51-434...| HD 209458 Formation|4755.956377260051...|       [[WXY-12345]]| 0.4966529856392779|[Ytterbium, Yttri...|{arctic, frozen, ...|  [{-3.143, 1.928, ...|                5|
|50bcbce3-1776-498...| HD 209458 Formation|1794.7130806

`transform` is one the important functions to manipulate arrays that you should know. It returns an array of elements after applying a transformation function to each element in the input array.

#### Step 4: Filter only rows whose `elements` column contains ``Ytterbium``

In [ ]:
# filter todo

+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+----------------------+
|                uuid|              system|  stardate_discovery|              flybys| habitability_index|            elements|     planet_taxonomy|planetary_temperatures|
+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+----------------------+
|33dd1b1a-cf51-434...| HD 209458 Formation|4755.956377260051...|       [[WXY-12345]]| 0.4966529856392779|[Ytterbium, Yttri...|{arctic, frozen, ...|  [{-3.143, 1.928, ...|
|1dc8fffb-862e-4e9...|     GJ 436 Ensemble|2596.324030514523...|[[IJK-12345], [NO...| 0.6565566352690925|[Manganese, Thuli...|{tropical, humid,...|  [{32.677, 31.896,...|
|f0a49362-f893-4f6...|Epsilon Eridani C...|8131.814979457421...|[[FGH-8765], [RST...|0.32636799741726885|[Thulium, Ytterbi...|{arctic, frozen, ..

#### Step 5: What if we want each array element on a single row?

In [ ]:
# todo

+--------------------+--------------------+------------+
|                uuid|  stardate_discovery|rare_element|
+--------------------+--------------------+------------+
|1856a72e-2fde-4f0...|4484.736272674565...|    Platinum|
|1856a72e-2fde-4f0...|4484.736272674565...|     Hafnium|
|1856a72e-2fde-4f0...|4484.736272674565...|     Rhodium|
|1856a72e-2fde-4f0...|4484.736272674565...|    Europium|
|33dd1b1a-cf51-434...|4755.956377260051...|   Ytterbium|
|33dd1b1a-cf51-434...|4755.956377260051...|     Yttrium|
|33dd1b1a-cf51-434...|4755.956377260051...|   Beryllium|
|33dd1b1a-cf51-434...|4755.956377260051...|     Rhodium|
|33dd1b1a-cf51-434...|4755.956377260051...|  Dysprosium|
|50bcbce3-1776-498...|1794.713080699246...|     Terbium|
|50bcbce3-1776-498...|1794.713080699246...|     Yttrium|
|729cb3af-41db-406...|3023.989300103815...|     Yttrium|
|729cb3af-41db-406...|3023.989300103815...|    Scandium|
|729cb3af-41db-406...|3023.989300103815...|      Erbium|
|729cb3af-41db-406...|3023.9893

#### Step 5.1: What about collecting the array?

In [ ]:
# TODO

+--------------------+--------------------+
|                uuid|       rare_elements|
+--------------------+--------------------+
|000018ef-90d8-400...|[Lanthanum, Vanad...|
|0003c524-66c1-48e...|[Rhodium, Niobium...|
|0005f457-38a6-432...|[Europium, Pallad...|
|00116cc6-2831-443...|[Niobium, Samariu...|
|0014aaa0-87b5-4bf...|[Niobium, Gadolin...|
|001629c8-8788-48b...|[Lanthanum, Gold,...|
|00219b06-5ab6-4d5...|[Tantalum, Scandi...|
|00258c69-3046-40b...|[Rhodium, Praseod...|
|002dd4f7-12c4-499...|[Ruthenium, Vanad...|
|003ba161-85c1-4c7...|            [Cobalt]|
|004c3df8-5270-4c3...|[Gold, Europium, ...|
|004f8fd5-a5b4-438...|[Zirconium, Lithium]|
|005a3b75-a03d-4f8...|     [Terbium, Gold]|
|0061af5d-0c22-472...|[Cerium, Gadolini...|
|0066f537-8330-4cd...|[Erbium, Terbium,...|
|006cb3cb-9d0a-44b...|[Promethium, Ceri...|
|00764a7f-7a49-4e2...|[Ytterbium, Gadol...|
|0095570f-fd7b-408...|[Tantalum, Chromi...|
|00a0c646-095e-496...|[Chromium, Tantal...|
|00a10f6d-bef5-448...|[Cerium, L

> There are dozens of functions that can be applied to array manipulation. These are the ones I find most useful, but you can take a look at the documentation and look for all functions specified as a "Collection function" in the description.

Here's a list of important array functions that you should be familiar with when working with Spark DataFrames:
1. `array()`: Create an array from multiple columns or values.
2. `concat()`: Concatenate multiple arrays.
3. `size()`: Get the size (length) of an array.
4. `element_at()`: Retrieve an element at a specific index in an array.
5. `slice()`: Extract a subarray from an array.
6. `array_contains()`: Check if an array contains a specific value.
7. `arrays_zip()`: Combine multiple arrays into a single array of structs.
8. `array_distinct()`: Remove duplicate elements from an array.
9. `array_except()`: Return an array containing elements from the first array that are not present in the second array.
10. `array_intersect()`: Return an array containing elements that are present in both input arrays.
11. `array_union()`: Return an array containing elements from both input arrays, without duplicates.
12. `array_remove()`: Remove a specific value from an array.
13. `array_sort()`:Sort the elements of an array in ascending order.
14. `array_max()`: Find the maximum value in an array.
15. `array_min()`: Find the minimum value in an array.
16. `array_position()`: Find the position (index) of a specific value in an array.
17. `array_repeat()`: Create an array by repeating a value for a specified number of times.
18. `flatten()`: Flatten a nested array structure.

#### Step 6: Create a UDF to calculate the size of an array

In [ ]:
# TODO function

###### Step 6.1: Register the UDF and use it

In [ ]:
# TODO

Now we can apply this functions to our sample `sparkDataFrame`:

In [ ]:
# TODO

+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+----------------------+-----------------+
|                uuid|              system|  stardate_discovery|              flybys| habitability_index|            elements|     planet_taxonomy|planetary_temperatures|num_rare_elements|
+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+----------------------+-----------------+
|1856a72e-2fde-4f0...|Proxima Centauri ...|4484.736272674565...|[[WXY-234], [STU-...|  0.648708376148531|[Platinum, Hafniu...|{savannah, dry, 0...|  [{30.301, 37.108,...|                4|
|33dd1b1a-cf51-434...| HD 209458 Formation|4755.956377260051...|       [[WXY-12345]]| 0.4966529856392779|[Ytterbium, Yttri...|{arctic, frozen, ...|  [{-3.143, 1.928, ...|                5|
|50bcbce3-1776-498...| HD 209458 Formation|1794.7130806

#### Step 6: Create a UDF to perform the operation of extracting the average daytime temperature

In [ ]:
# TODO 

In [ ]:
# TODO

UDF execution time: 5.804316 seconds


#### Step 7: Define a Pandas UDF to obtain the moving average of the daytime temperatures for each planet

In [ ]:
import pandas as pd
import pyspark.sql.types as tp
from pyspark.sql.types import FloatType

# TODO

+--------------------+-----------------+--------------+
|                uuid|num_rare_elements|moving_average|
+--------------------+-----------------+--------------+
|48a2ac50-f61d-422...|                4|          NULL|
|0e822259-3565-460...|                7|          NULL|
|ca734eff-ce27-435...|                7|           6.0|
|a5af22a3-f575-4c3...|                6|     6.6666665|
|f5eb74cc-af18-4f4...|                2|           5.0|
+--------------------+-----------------+--------------+



##### Step 8: Define a UDF to calculate the habitability score of each planet based on its average daytime temperature and the number of rare elements it contains. The habitability score can be calculated using the following formula:

```
habitability_score = (average_daytime_temperature / 100) * (1 - (number_of_rare_elements / 10))
``` 


In [ ]:
# TODO

In [ ]:
# TODO

In [ ]:
# We can now compute our average habitability using pandas_in_spark
# TODO

+---------------+----------------+----------------+
|    star_system|environment_type|avg_habitability|
+---------------+----------------+----------------+
|GJ 436 Ensemble|          alpine|       0.4834973|
|GJ 436 Ensemble|          arctic|       0.5150751|
|GJ 436 Ensemble|            arid|      0.49071103|
|GJ 436 Ensemble|     continental|       0.5113722|
|GJ 436 Ensemble|          desert|      0.51230043|
+---------------+----------------+----------------+



#### Step 7: What is the most habitable star system overall?

In [ ]:
# TODO

+--------------------+-----------------------+
|         star_system|habitability_per_system|
+--------------------+-----------------------+
|      WASP-12 System|     0.5101965169111887|
|    TRAPPIST-1 Array|     0.5048847595850626|
| 55 Cancri Formation|      0.502709964911143|
|    Kepler-186 Array|     0.5014570322301652|
|     GJ 436 Ensemble|     0.5011172162161933|
|  PSR B1257+12 Array|      0.500620143281089|
|     Kepler-22 Array|      0.500020424524943|
|    Luhman 16 Binary|     0.5000161594814725|
| Wolf 1061 Formation|     0.4997712042596605|
| Gliese 581 Ensemble|    0.49846984611617196|
| HD 209458 Formation|    0.49811175134446883|
|Proxima Centauri ...|     0.4978453483846452|
|   Kepler-452 System|     0.4976225992043813|
|  Tau Ceti Formation|    0.49714238776101005|
|        Vega Cluster|    0.49698305792278713|
|      HR 8799 System|     0.4966701302263472|
|     GJ 1214 Cluster|     0.4962238040235307|
|   HD 40307 Ensemble|     0.4962113897005717|
|Epsilon Erid